# Cut the Kade pages into lines with kraken

Step 1 of the Kade LoRA plan. Every Kade page is cut into one image per line of writing, so the
Qwen3-VL Kurrent adapter (which was trained on single lines) can read them, and so each line image
can be paired with its line of the label for fine-tuning.

Set up your computer first with `docs/ubuntu-setup.md`, and pick the **Badger Scribe** kernel
(top right). Then run the cells from top to bottom.

**What comes out:** in your data folder, `kade_lines/<page>/000.png, 001.png, ...` (one image per
line, top to bottom) and `kade_lines/line_counts.csv`, which says for each page whether kraken found
the same number of lines as the label has. Only pages where they match can be used for training at
first.

## 1. Settings

`BADGER_DRIVE` is the folder that holds `images/` and `train.csv`. The setup guide sets it for you; change it here only if yours is somewhere else.

`LIMIT` is how many pages to cut. Start with 5, look at the crops, then set it to `None` for all of them.

In [ ]:
import csv
import os
import time
from pathlib import Path

from PIL import Image

DRIVE = Path(os.environ.get("BADGER_DRIVE", Path.home() / "badger-scribe-data"))
IMAGES = DRIVE / "images"
LABELS = DRIVE / "train.csv"
LINES = DRIVE / "kade_lines"

LIMIT = 5        # None = every Kade page
DEVICE = "cpu"   # "cuda" if this computer has an NVIDIA GPU with CUDA PyTorch

## 2. Check the setup

This should print the kraken version and how many Kade images it found. If it stops with an error, the message says what to fix.

In [ ]:
import importlib.metadata
import kraken  # if this fails, the kernel isn't "Badger Scribe" or the install step was skipped

print("kraken", importlib.metadata.version("kraken"))
if not IMAGES.is_dir():
    raise SystemExit(f"Can't find {IMAGES}. Set BADGER_DRIVE to the folder with images/ and train.csv.")
if not LABELS.is_file():
    raise SystemExit(f"Can't find {LABELS}.")
pages = sorted(IMAGES.glob("kade_*.jpg"))
print(f"{len(pages)} Kade images in {IMAGES}")
if not pages:
    raise SystemExit("No kade_*.jpg images yet. Copy them from the shared drive (setup guide, step 4).")

## 3. Cut the pages into lines

About 12 seconds a page on a laptop CPU, so all 436 Kade pages take about 1.5 hours. Pages that already have a folder in `kade_lines` are skipped, so you can stop and re-run.

In [ ]:
from kraken import blla
from kraken.lib.segmentation import extract_polygons

labels = {r["page_id"]: r["text"] for r in csv.DictReader(open(LABELS, encoding="utf-8-sig", newline=""))}
LINES.mkdir(exist_ok=True)
todo = pages if LIMIT is None else pages[:LIMIT]

for n, path in enumerate(todo, 1):
    out = LINES / path.stem
    if out.is_dir():
        continue
    start = time.time()
    im = Image.open(path).convert("RGB")
    seg = blla.segment(im, device=DEVICE)  # default model; lines come back in reading order
    tmp = LINES / (path.stem + ".partial")
    tmp.mkdir(exist_ok=True)
    for i, (crop, _) in enumerate(extract_polygons(im, seg)):
        crop.save(tmp / f"{i:03d}.png")
    tmp.rename(out)  # only a finished page gets its real folder name
    print(f"[{n}/{len(todo)}] {path.stem}: {len(seg.lines)} lines in {time.time() - start:.0f}s")

## 4. Compare with the labels

Counts the lines in each page's label, leaving out the `Postmark ...` line (the postmark is on the other side of the card, so it isn't in the image). Test pages have no label and show 0.

In [ ]:
rows = []
for out in sorted(p for p in LINES.iterdir() if p.is_dir() and not p.name.endswith(".partial")):
    ref = [l for l in labels.get(out.name, "").splitlines()
           if l.strip() and not l.startswith("Postmark")]
    found = len(list(out.glob("*.png")))
    rows.append({"page_id": out.name, "kraken_lines": found, "label_lines": len(ref),
                 "match": found == len(ref)})

with open(LINES / "line_counts.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["page_id", "kraken_lines", "label_lines", "match"])
    writer.writeheader()
    writer.writerows(rows)

labeled = [r for r in rows if r["label_lines"]]
print(f"{sum(r['match'] for r in labeled)} of {len(labeled)} labeled pages have matching line counts")
for r in labeled:
    if not r["match"]:
        print(f"  {r['page_id']}: kraken {r['kraken_lines']}, label {r['label_lines']}")

## 5. Look at one page

Check by eye that each crop is one whole line: not two lines merged, not half a line.

In [ ]:
from IPython.display import display

page = rows[0]["page_id"]  # or type a page name, e.g. "kade_112_p004"
for crop in sorted((LINES / page).glob("*.png")):
    print(crop.name)
    display(Image.open(crop))